# Notre monde calculé — notebook collectif

Ce notebook de référence permet à la classe de choisir son étoile, sa planète et ses boucles, calculer une évolution longue et produire automatiquement un dossier court avec six graphiques. Il fonctionne seul dans le navigateur. Il utilise la réduction géologique fournie : il ne remplace pas les 13 routines que les groupes doivent écrire. Aucun téléchargement de code distant.

**Attention :** paramètres fictifs, lois simplifiées, équilibre climatique rapide. Pas variable croissant, sans contrôle local automatique par rejet. Une température tempérée ne suffit pas à démontrer une habitabilité.

## Nos paramètres et nos prédictions

Les formules S = L/(4πd²), g = GM/R² et A = 4πR² sont fournies. Les stocks et les lois sont décrits dans les fiches collectives. Choisir une expérience à la fois ; annoncer l’effet attendu avant de lancer le calcul.

In [ ]:
# À LA CLASSE : choisir votre monde. Grandeurs et lois dans les fiches.
monde = {
    'nom': 'Asteria',
    'luminosite_W': 3.828e26,
    'distance_m': 1.496e11,
    'masse_kg': 5.97e24,
    'rayon_m': 6.371e6,
    'duree_yr': 5e9,
    'stellar_validity_yr': 1e10,  # hypothèse de validité, pas une durée de vie calculée
    'stellar_initial_ratio': .75,
    'stellar_final_ratio': 1.,
    'water_kg': 1.4e21,
    'co2_initial_mol': 3e19,
    'n2_mol': 1.39e20
}
# Chaque case active une hypothèse précise.
boucles = {'weather_feedback': True, 'ice_feedback': True,
           'water_feedback': True, 'solar_evolution': True}


## Moteur de référence fourni

Exécuter cette cellule telle quelle. Les relations sont visibles dans les fonctions ; les élèves les expliquent à partir des fiches.

In [ ]:
"""Maquette de temps long : eau conservée, carbone conservé, climat rapide.
Les fonctions sont courtes et les hypothèses explicites. Ce n'est pas une
reconstruction de la Terre. Les flux d'eau rapides sont intégrés exactement.
"""
from math import exp, isfinite

DUREE = 4.031e9  # début de l'Archeen arrondi, jusqu'au présent
PARAMS = {'water_kg': 1.4e21, 'carbon_mol': 1e23,
          'co2_initial_mol': 3e19, 'volcanic_mol_yr': 1e13,
          'weather_mol_yr': 1e13, 'co2_ref_mol': 7.49e16,
          'pref_Pa': 42., 'rain_ref_kg_yr': 5e17,
          'vapor_tau_yr': .02, 'area_m2': 5.10e14,
          'weather_feedback': True, 'ice_feedback': True,
          'water_feedback': True, 'solar_evolution': True}

def climat(co2, vapor, soleil, precedent, p):
    """Climat rapide à CO2 fixé ; la branche dépend de l'état précédent."""
    T = precedent
    for _ in range(400):
        ice = max(0., min(1., (283-T)/20)) if p['ice_feedback'] else 0.
        albedo = .30 + .30*ice
        teq = (soleil*(1-albedo)/4/5.67e-8)**.25
        # Approximation de la pression de CO2 : fond N2 moderne fixe.
        mass = co2*.044 + p.get('n2_mol',1.39e20)*.028 + vapor
        P = mass*p.get('gravity_m_s2',9.81)/p['area_m2']
        pco2 = P*co2/(co2+p.get('n2_mol',1.39e20)+vapor/.018)
        serre = 66*pco2/(pco2+p['pref_Pa'])
        cible = teq+serre
        if abs(cible-T) < 1e-8:
            return cible, ice, pco2, albedo
        T = .5*(T+cible)
    raise ValueError('Climat rapide non convergé')

def hydrologie(T, ice, vapor, dt, p):
    """Atmosphère <-évaporation-- océan ; atmosphère --pluie/neige-> surface.
    Le stock vapeur suit dV/dt=E-V/tau, intégré exactement. Les précipitations
    sont obtenues par bilan, sans des milliards de petits pas annuels.
    La fraction ice partitionne la surface eau/glace, sans créer d'eau.
    """
    ouvert = max(0., 1-ice)
    facteur = max(.05, min(3., exp((T-288)/30)))
    E = p['rain_ref_kg_yr']*ouvert*facteur
    cible = min(E*p['vapor_tau_yr'], p['water_kg'])
    nouveau = cible+(vapor-cible)*exp(-dt/p['vapor_tau_yr'])
    P = E-(nouveau-vapor)/dt
    surface = p['water_kg']-nouveau
    return {'vapor_kg':nouveau,'ocean_kg':surface*(1-ice),
            'ice_kg':surface*ice,'evap_kg_yr':E,
            'precip_kg_yr':P,'rain_mm_yr':P*(1-ice)/p['area_m2'],
            'snow_mm_yr':P*ice/p['area_m2']}

def carbone(C, manteau, roches, T, ice, rain, land, dt, p):
    """dC/dt=F-kC ; solution exacte pour k et F figés sur le pas.
    Transfert identique vers les roches, et prélèvement dans le manteau.
    """
    F = min(p['volcanic_mol_yr'], manteau/dt)
    if p['weather_feedback']:
        chaleur = max(0., 1+(T-288)/50)
        pluie = rain/p['rain_ref_kg_yr'] if p['water_feedback'] else 1.
        k = p['weather_mol_yr']/p['co2_ref_mol']*land*chaleur*pluie*(1-ice)
    else:
        k = 0.
    if k > 0:
        r = exp(-k*dt)
        nouveau = C*r+F/k*(1-r)
    else:
        nouveau = C+F*dt
    transfert = C+F*dt-nouveau
    return nouveau, manteau-F*dt, roches+transfert, transfert/dt

def simuler(p=None, dt=1e6, duree=DUREE):
    p = dict(PARAMS, **(p or {}))
    if not isfinite(dt) or dt<=0 or not 0<duree<=p.get('stellar_validity_yr',1e10):
        raise ValueError('Pas/durée invalide')
    for key in ['water_kg','carbon_mol','co2_ref_mol','pref_Pa','vapor_tau_yr','area_m2','rain_ref_kg_yr']:
        if not isfinite(p[key]) or p[key]<=0: raise ValueError('Paramètre positif requis : '+key)
    if not 0<=p['co2_initial_mol']<=p['carbon_mol']: raise ValueError('Stock initial CO2 invalide')
    for key in ['volcanic_mol_yr','weather_mol_yr']:
        if not isfinite(p[key]) or p[key]<0: raise ValueError('Flux invalide')
    C=p['co2_initial_mol']; M=p['carbon_mol']-C; R=0.; V=1e16; T=300.; t=0.
    h=[]
    pas_adaptatif=min(dt,100.)
    while True:
        fraction=t/duree
        flux=p.get('stellar_flux_W_m2',1361.)
        debut=p.get('stellar_initial_ratio',.75)
        fin=p.get('stellar_final_ratio',1.)
        soleil=flux*(debut+(fin-debut)*fraction) if p['solar_evolution'] else flux
        land=.15+.85*fraction  # efficacité relative d'altération, hypothèse imposée
        T,ice,pc,A=climat(C,V,soleil,T,p)
        # Diagnostics instantanés de pluie E=V/tau après relaxation rapide.
        eau=hydrologie(T,ice,V,1.,p)
        point={'time_yr':t,'age_Ga':(duree-t)/1e9,'T_K':T,'ice':ice,
               'co2_mol':C,'mantle_c_mol':M,'rock_c_mol':R,
               'pco2_Pa':pc,'albedo':A,'solar_W_m2':soleil,'land_factor':land,
               'vapor_kg':V,'ocean_kg':(p['water_kg']-V)*(1-ice),
               'ice_kg':(p['water_kg']-V)*ice,
               'evap_kg_yr':eau['evap_kg_yr'],
               'precip_kg_yr':V/p['vapor_tau_yr'],
               'rain_mm_yr':V/p['vapor_tau_yr']*(1-ice)/p['area_m2'],
               'snow_mm_yr':V/p['vapor_tau_yr']*ice/p['area_m2']}
        for k,v in point.items():
            if not isfinite(v) or v < -1e-6: raise ValueError('État invalide : '+k)
        h.append(point)
        if t>=duree: break
        pas=min(pas_adaptatif,duree-t)
        pas_adaptatif=min(dt,pas_adaptatif*1.05)
        fluxeau=hydrologie(T,ice,V,pas,p)
        C,M,R,W=carbone(C,M,R,T,ice,fluxeau['precip_kg_yr'],land,pas,p)
        # Moyennes sur [t,t+pas] séparées des diagnostics instantanés.
        point['interval_evap_kg_yr']=fluxeau['evap_kg_yr']
        point['interval_precip_kg_yr']=fluxeau['precip_kg_yr']
        point['interval_weather_mol_yr']=W
        V=fluxeau['vapor_kg']; t+=pas
    return h


## Fabrication automatique du dossier

Le rapport est fabriqué à partir des états calculés et de vos paramètres. Il distingue les choix, les résultats et les hypothèses.

In [ ]:
"""Un dossier court issu des états réellement calculés, sans service externe."""
from pathlib import Path
from html import escape
from math import log10, pi
import json


def nombre(v):
    if v == 0: return '0'
    if abs(v)>=1e5 or abs(v)<.001: return f'{v:.3g}'
    return f'{v:.3f}'.rstrip('0').rstrip('.')

def graphique(h, cle, titre, unite, facteur=1., offset=0., logarithme=False):
    valeurs=[s[cle]/facteur+offset for s in h]
    if logarithme:
        if min(valeurs)<=0: raise ValueError('Axe logarithmique : valeur non positive')
        valeurs=[log10(v) for v in valeurs]
    lo,hi=min(valeurs),max(valeurs)
    marge=max((hi-lo)*.08,abs(hi)*1e-6,1e-6)
    lo-=marge;hi+=marge
    if cle=='ice':lo,hi=0.,100.
    D=h[-1]['time_yr'] or 1
    points=' '.join(f'{80+400*s["time_yr"]/D:.2f},{190-130*(v-lo)/(hi-lo):.2f}' for s,v in zip(h,valeurs))
    axes=[]
    for j in range(4):
        y=190-130*j/3;v=lo+(hi-lo)*j/3
        label=nombre(10**v if logarithme else v)
        if cle=='ocean_kg': label=f'{v:.8f}'
        axes.append(f'<path d="M80 {y}H480" stroke="#d7e0e5"/><text x="73" y="{y+5}" text-anchor="end">{label}</text>')
    for j in range(5):
        x=80+400*j/4
        axes.append(f'<text x="{x}" y="215" text-anchor="middle">{nombre(D/1e9*j/4)}</text>')
    return f'''<svg viewBox="0 0 520 250" role="img" aria-label="{escape(titre)} en {escape(unite)}">
    <title>{escape(titre)}</title><text x="80" y="28">{escape(titre)} ({escape(unite)})</text>
    {''.join(axes)}<polyline points="{points}" fill="none" stroke="#126478" stroke-width="2"/>
    <text x="165" y="245">Temps depuis le début (Ga)</text></svg>'''

def generer(fichier=None):
    base=Path('resultats')
    fichier=Path(fichier) if fichier else base/'histoire_complete.json'
    data=json.loads(fichier.read_text())
    monde=data['monde'];p=dict(PARAMS,**data.get('params',{}));h=data['history']
    if not h: raise ValueError('Historique vide')
    if any(b['time_yr']<=a['time_yr'] for a,b in zip(h,h[1:])):raise ValueError('Temps non croissants')
    debut,fin=h[0],h[-1];D=fin['time_yr']-debut['time_yr']
    eau0=sum(debut[k] for k in ['vapor_kg','ocean_kg','ice_kg'])
    carbone0=sum(debut[k] for k in ['co2_mol','mantle_c_mol','rock_c_mol'])
    err_eau=max(abs(sum(s[k] for k in ['vapor_kg','ocean_kg','ice_kg'])-eau0)/max(1,eau0) for s in h)
    err_carbone=max(abs(sum(s[k] for k in ['co2_mol','mantle_c_mol','rock_c_mol'])-carbone0)/max(1,carbone0) for s in h)
    gelee=sum(s['ice']>=.9 for s in h)
    resume=(f"La température passe de {nombre(debut['T_K']-273.15)} °C à {nombre(fin['T_K']-273.15)} °C. "
            f"Elle reste entre {nombre(min(s['T_K'] for s in h)-273.15)} et {nombre(max(s['T_K'] for s in h)-273.15)} °C. "
            f"La fraction de glace finale vaut {nombre(100*fin['ice'])} %. "
            + ('Aucun état enregistré ne dépasse 90 % de glace.' if gelee==0 else f'{gelee} états enregistrés dépassent 90 % de glace ; leur nombre n’est pas une durée.'))
    if monde['distance_m']<=0:raise ValueError('Distance invalide')
    caracteristiques=[('Étoile : luminosité de référence',nombre(monde['luminosite_W'])+' W'),
      ('Luminosité / référence solaire',nombre(monde['luminosite_W']/3.828e26)),
      ('Distance orbitale',nombre(monde['distance_m']/1.496e11)+' UA'),
      ('Masse planétaire',nombre(monde['masse_kg'])+' kg'),
      ('Rayon planétaire',nombre(monde['rayon_m']/1000)+' km'),
      ('Gravité calculée',nombre(6.67e-11*monde['masse_kg']/monde['rayon_m']**2)+' m/s²'),
      ('Durée calculée',nombre(D/1e9)+' Ga'),
      ('Flux reçu : début → fin',nombre(debut['solar_W_m2'])+' → '+nombre(fin['solar_W_m2'])+' W/m²')]
    evolution=[('CO₂ atmosphérique','pco2_Pa','Pa'),('Pluie moyenne','rain_mm_yr','mm/an'),
      ('Océan liquide','ocean_kg','kg'),('Vapeur atmosphérique','vapor_kg','kg'),('Eau glacée','ice_kg','kg')]
    lignes=[(nom,nombre(debut[k]),nombre(fin[k]),unit) for nom,k,unit in evolution]
    bilan=f"Erreurs relatives maximales : eau {err_eau:.2e} ; carbone {err_carbone:.2e}. Ces bilans contrôlent les transferts du modèle."
    # Une comparaison de pas n'est attribuée à ce monde que si les états concordent.
    convergence='Comparaison de pas non disponible pour cette simulation.'
    cf=fichier.parent/'histoire_comparaisons.json'
    if cf.exists():
        c=json.loads(cf.read_text())
        if c.get('initial')==debut and c.get('final')==fin:
            convergence=f"Écart maximal de température pour des plafonds de pas successivement divisés par deux : {c['max_delta_T_half_K']:.4g} K puis {c['max_delta_T_quarter_K']:.4g} K."
    assumptions=('Les paramètres sont choisis, pas mesurés pour une planète réelle. L’étoile suit une loi linéaire de luminosité prescrite, valable sur la durée déclarée ; son spectre et sa durée de vie ne sont pas calculés. '
      'Le climat est un équilibre rapide avec une loi de serre bornée. Pluie et évaporation sont des moyennes planétaires. L’eau totale est conservée ; la fraction de glace sert aussi à partager la masse d’eau de surface. '
      'Nuages, météo, chaleur latente, vivant, tectonique détaillée et chimie complète sont absents. Un état tempéré ne démontre pas l’habitabilité.')
    table=lambda rows:'<table>'+''.join('<tr>'+''.join('<td>'+escape(str(v))+'</td>' for v in row)+'</tr>' for row in rows)+'</table>'
    graphs=''.join(graphique(h,*args) for args in [
      ('T_K','Température','°C',1,-273.15,False),('pco2_Pa','CO₂ atmosphérique','Pa — axe log',1,0,True),
      ('rain_mm_yr','Pluie moyenne','mm/an',1,0,False),('solar_W_m2','Flux stellaire reçu','W/m²',1,0,False),
      ('ocean_kg','Océan liquide','10²¹ kg',1e21,0,False),('ice','Fraction de glace','%',.01,0,False)])
    title='Notre monde calculé '+str(monde['nom'])
    html=f'''<!doctype html><html lang="fr"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>{escape(title)}</title>
    <style>body{{font:16px/1.45 system-ui;color:#193442;max-width:1000px;margin:30px auto;padding:0 22px}}h1,h2{{line-height:1.15}}h2{{font-size:21px;margin-top:25px}}table{{border-collapse:collapse;width:100%;font-size:14px}}td{{padding:5px 9px;border-bottom:1px solid #dde5e8}}td:first-child{{font-weight:600}}.graphs{{display:grid;grid-template-columns:1fr 1fr;gap:8px}}svg{{width:100%;font:12px system-ui}}button{{font:inherit;padding:8px 14px}}.note{{color:#526876;font-size:14px}}@media(max-width:650px){{.graphs{{grid-template-columns:1fr}}}}@media print{{@page{{size:A4;margin:15mm}}body{{font-size:11px;margin:0;padding:0}}h1{{font-size:23px}}h2{{font-size:16px}}td{{font-size:10px;padding:3px 6px}}button,.navigation{{display:none}}svg{{break-inside:avoid}}.graphs{{break-before:page;grid-template-columns:1fr 1fr}}.limits{{break-before:page}}}}</style>
    <h1>{escape(title)}</h1><p>Une synthèse courte des résultats de notre simulation · {nombre(D/1e9)} milliards d’années</p>
    <button onclick="window.print()">Imprimer ou enregistrer en PDF</button><p class="navigation"><a href="planete_systemes.html">Explorer le graphe et les boucles</a></p>
    <h2>Le monde choisi</h2>{table(caracteristiques)}
    <h2>Ce que le calcul donne</h2><p>{escape(resume)}</p>
    {table([('Grandeur','Début','Fin','Unité')]+lignes)}
    <p class="note">Le début est l’état choisi avant sa relaxation. Les paramètres de référence solaire sont des unités de comparaison ; aucune chronologie terrestre n’est imposée.</p>
    <div class="graphs">{graphs}</div>
    <div class="limits"><h2>Comment interpréter ces courbes</h2><p>Comparer les variations du CO₂, du rayonnement reçu et de la température. La boucle d’altération peut compenser une partie du changement du rayonnement : température → pluie → altération → retrait du CO₂ → effet de serre. La boucle glace → albédo → refroidissement amplifie un état froid lorsqu’elle est sollicitée. Vérifier ces explications en désactivant une boucle dans le graphe ; une seule courbe ne prouve pas une causalité.</p>
    <h2>Contrôles numériques</h2><p>{escape(bilan)}</p><p>{escape(convergence)}</p>
    <h2>Les hypothèses à garder en tête</h2><p>{escape(assumptions)}</p>
    <h2>Conclusion de la classe</h2><p>Notre monde est… ; les deux mécanismes qui expliquent le mieux son évolution sont… ; la comparaison qui soutient notre explication est… ; notre principale limite est…</p>
    <p class="note">Données sources : {escape(fichier.name)}. Généré automatiquement à partir des états enregistrés. Les élèves complètent l’interprétation, pas les résultats calculés.</p></div></html>'''
    dossier=fichier.parent;out=dossier/'synthese_monde.html';out.write_text(html)
    md='# '+title+'\n\n'+resume+'\n\n## Le monde choisi\n\n'+''.join('- '+a+' : '+b+'\n' for a,b in caracteristiques)
    md+='\n## Contrôles\n\n'+bilan+'\n\n'+convergence+'\n\n## Hypothèses\n\n'+assumptions+'\n\nLes six graphiques sont dans synthese_monde.html, imprimable en PDF.\n'
    (dossier/'synthese_monde.md').write_text(md)
    return out



In [ ]:
from math import pi
for cle in ['luminosite_W','distance_m','masse_kg','rayon_m','duree_yr','water_kg']:
    assert monde[cle]>0, cle
assert monde['duree_yr']<=monde['stellar_validity_yr']
p=dict(PARAMS,**boucles)
for cle in ['stellar_validity_yr','stellar_initial_ratio','stellar_final_ratio',
            'water_kg','co2_initial_mol','n2_mol']:p[cle]=monde[cle]
p['stellar_flux_W_m2']=monde['luminosite_W']/(4*pi*monde['distance_m']**2)
p['gravity_m_s2']=6.67e-11*monde['masse_kg']/monde['rayon_m']**2
p['area_m2']=4*pi*monde['rayon_m']**2
h=simuler(p,dt=1e6,duree=monde['duree_yr'])
resultats=Path('resultats');resultats.mkdir(exist_ok=True)
fichier=resultats/'notre_monde.json'
fichier.write_text(json.dumps({'monde':monde,'params':p,'history':h},ensure_ascii=False))
rapport=generer(fichier)
print(monde['nom'],len(h),'états ; température finale',round(h[-1]['T_K']-273.15,1),'°C')
print('Rapport créé :',rapport)


In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
page=rapport.read_text()
display(HTML('<iframe title="Synthèse du monde" style="width:100%;height:1000px;border:0" sandbox="allow-scripts allow-modals" srcdoc="'+escape(page,quote=True)+'"></iframe>'))
# Les fichiers se téléchargent depuis le navigateur, puis se déposent dans OneDrive.
for chemin,nom in [(rapport,'Synthese_'+monde['nom']+'.html'),(fichier,'Resultats_'+monde['nom']+'.json')]:
    contenu=b64encode(chemin.read_bytes()).decode('ascii')
    display(HTML('<p><a download="'+escape(nom)+'" href="data:application/octet-stream;base64,'+contenu+'">Télécharger '+escape(nom)+'</a></p>'))


## À la classe — compléter la synthèse

Notre hypothèse avant calcul : …

Le résultat qui nous surprend : …

La boucle qui explique le mieux notre monde : …

L’expérience de comparaison qui soutient cette explication : …

La limite à annoncer : …

Télécharger le rapport HTML et le notebook complété puis les déposer dans OneDrive. Ouvrir le rapport HTML téléchargé dans un navigateur pour l’imprimer/enregistrer en PDF. Le navigateur ne synchronise pas automatiquement avec OneDrive.